In [ ]:
from datasets import load_dataset, Audio
import itertools
import pandas as pd
import soundfile as sf
import io

ds = load_dataset(
    "dsfsi-anv/za-african-next-voices",
    "zul",
    split="train",
    streaming=True,
)

# Skip decoding the waveform array
ds = ds.cast_column("audio", Audio(decode=False))
NUM_SAMPLES = 5000
top5 = list(itertools.islice(ds, NUM_SAMPLES))

# Flatten audio column — drop bytes, keep only path
def flatten(row):
    flat = {}
    for col, val in row.items():
        if isinstance(val, dict) and "bytes" in val:
            flat[f"{col}_path"] = val.get("path")
            try:
                with sf.SoundFile(io.BytesIO(val["bytes"])) as f:
                    flat[f"{col}_duration_seconds"] = round(len(f) / f.samplerate, 2)
                    flat[f"{col}_sampling_rate"] = f.samplerate
                    flat[f"{col}_channels"] = f.channels
            except Exception:
                flat[f"{col}_duration_seconds"] = None
        else:
            flat[col] = val
    return flat

df = pd.DataFrame([flatten(row) for row in top5])

In [ ]:
df

In [ ]:
# Build metadata summary
summary = pd.DataFrame({
    "dtype":       df.dtypes,
    "null_count":  df.isnull().sum(),
    "null_%":      (df.isnull().sum() / len(df) * 100).round(2),
    "unique":      df.nunique(),
    "sample_val":  df.iloc[0],
})

print(f"Sampled {len(df)} rows | {len(df.columns)} columns\n")

In [ ]:
summary